# 규제화 회귀 — 릿지·라소·엘라스틱넷

**자료**.  Cereals.csv — 77 종의 시리얼, 영양 성분 12 개 + 소비자 등급 *rating*.

**목표**.  4 부의 다중공선성 문제(*VIF* 가 10 을 넘는 변수 다섯) 를 해결하는 세 가지 규제화 도구를 차례로 적용한다.

1. **릿지**(Ridge) — L² 벌점.  닫힌 해 *β̂* = (*X*ᵀ*X* + *λI*)⁻¹*X*ᵀ*y*.  모든 계수를 부드럽게 0 으로 수축.
2. **라소**(Lasso) — L¹ 벌점.  좌표하강법.  일부 계수를 정확히 0 으로 — 변수 선택.
3. **엘라스틱넷**(Elastic Net) — L¹ + L² 혼합.  두 모수 (α, λ) 의 격자에서 교차검증.

모든 비교를 같은 표준화된 자료에서 수행한다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.model_selection import cross_val_score, KFold
from statsmodels.stats.outliers_influence import variance_inflation_factor

plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = '#FAF6F1'

## 1. 자료 불러오기

Pyodide(브라우저)·Colab·로컬 Python 모두에서 같은 한 줄로 동작.

In [ ]:
URL = ('https://raw.githubusercontent.com/leina99-lab/classes/main/'
       'AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/PCA/Cereals.csv')
try:
    from pyodide.http import open_url
    df = pd.read_csv(open_url(URL)).dropna()
except ImportError:
    df = pd.read_csv(URL).dropna()

features = ['calories','protein','fat','sodium','fiber','carbo','sugars',
            'potass','vitamins','shelf','weight','cups']
X_raw = df[features].values
y     = df['rating'].values

# 표준화 — 모든 변수의 평균 0, 표준편차 1
sx, sy = StandardScaler(), StandardScaler()
Xs = sx.fit_transform(X_raw)
ys = sy.fit_transform(y.reshape(-1, 1)).ravel()

print(f'표본 수 n = {len(df)},  변수 수 p = {len(features)}')
print(f'표준화 후 X 의 평균: {np.abs(Xs.mean(0)).max():.2e}  (≈ 0)')
print(f'표준화 후 X 의 표준편차: {Xs.std(0, ddof=0)}')

## 2. OLS 의 다중공선성 진단

규제화에 들어가기 전에 — 왜 OLS 가 안 되는지를 한 번 확인한다.
*VIF* 가 10 을 넘는 변수가 다섯이라는 것이 4 부의 결론이었다.

In [ ]:
# OLS 표준화 계수
ols = LinearRegression().fit(Xs, ys)
print('OLS 표준화 계수 (절댓값 순):')
order = np.argsort(np.abs(ols.coef_))[::-1]
for j in order:
    print(f'  {features[j]:>10s}  {ols.coef_[j]:+.4f}')

print('\nVIF:')
for j, name in enumerate(features):
    vif = variance_inflation_factor(Xs, j)
    flag = '   ← 강한 다중공선성' if vif > 10 else ''
    print(f'  {name:>10s}  {vif:6.2f}{flag}')

## 3. 릿지 — L² 벌점

손실함수.

$$\mathcal{L}_{\text{ridge}}(\beta) \;=\; \|y - X\beta\|^2 \;+\; \lambda\,\|\beta\|_2^2$$

닫힌 해.

$$\hat{\beta}_{\text{ridge}} \;=\; (X^T X \;+\; \lambda I)^{-1}\, X^T y$$

*λ* 가 커지면 모든 계수가 0 으로 끌려가되, 정확히 0 이 되지는 않는다.

In [ ]:
lams = np.logspace(-3, 4, 80)
ridge_path = np.zeros((len(lams), len(features)))
for i, lam in enumerate(lams):
    ridge_path[i] = Ridge(alpha=lam).fit(Xs, ys).coef_

# 5-fold CV 로 최적 λ
mses = []
for lam in lams:
    s = cross_val_score(Ridge(alpha=lam), Xs, ys, cv=5,
                        scoring='neg_mean_squared_error').mean()
    mses.append(-s)
mses = np.array(mses)
lam_best = lams[np.argmin(mses)]
print(f'최적 λ (Ridge) = {lam_best:.5f}')

In [ ]:
# 계수 경로 그림
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.set_facecolor('#FAF6F1')
cmap = plt.cm.tab20(np.linspace(0, 1, len(features)))
for j in range(len(features)):
    ax.plot(lams, ridge_path[:, j], color=cmap[j], linewidth=2.0, label=features[j])
ax.axvline(lam_best, color='#C0392B', linewidth=1.2, linestyle='--', alpha=0.7,
           label=f'CV 최적 λ = {lam_best:.4f}')
ax.set_xscale('log')
ax.set_xlabel('규제 강도  λ  (로그 축)')
ax.set_ylabel('표준화 계수 β*')
ax.set_title('Ridge — 계수 경로')
ax.legend(loc='upper right', frameon=False, fontsize=9, ncol=2)
ax.grid(alpha=0.2, linestyle='--')
plt.show()

### 3.1 결과 해석

*λ* 가 매우 작을 때(왼쪽 끝) 의 계수는 OLS 와 같다. *λ* 가 커지면서 *VIF* 가 큰 변수 — 칼로리·설탕·탄수화물 — 의 계수가 가장 먼저 진정되고, *VIF* 가 작은 변수 — 식이섬유·단백질 — 의 계수가 가장 오래 살아남는다. 어느 변수도 정확히 0 이 되지는 않는다 — L² 벌점의 정체성이다.

## 4. 라소 — L¹ 벌점

손실함수.

$$\mathcal{L}_{\text{lasso}}(\beta) \;=\; \|y - X\beta\|^2 \;+\; \lambda\,\|\beta\|_1$$

L¹ 벌점은 *βⱼ* = 0 자리에서 "꺾여" 있어, 좌표하강법으로 푼다. 일부 계수가 정확히 0 으로 떨어지는 "변수 선택" 효과가 자동으로 일어난다.

In [ ]:
lams_l = np.logspace(-4, 0.5, 80)
lasso_path = np.zeros((len(lams_l), len(features)))
for i, lam in enumerate(lams_l):
    lasso_path[i] = Lasso(alpha=lam, max_iter=20000).fit(Xs, ys).coef_

mses_l = []
for lam in lams_l:
    s = cross_val_score(Lasso(alpha=lam, max_iter=20000), Xs, ys, cv=5,
                        scoring='neg_mean_squared_error').mean()
    mses_l.append(-s)
mses_l = np.array(mses_l)
lam_best_l = lams_l[np.argmin(mses_l)]
print(f'최적 λ (Lasso) = {lam_best_l:.5f}')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.set_facecolor('#FAF6F1')
for j in range(len(features)):
    ax.plot(lams_l, lasso_path[:, j], color=cmap[j], linewidth=2.0, label=features[j])
ax.axvline(lam_best_l, color='#C0392B', linewidth=1.2, linestyle='--', alpha=0.7,
           label=f'CV 최적 λ = {lam_best_l:.4f}')
ax.set_xscale('log')
ax.set_xlabel('규제 강도  λ  (로그 축)')
ax.set_ylabel('표준화 계수 β*')
ax.set_title('Lasso — 계수가 정확히 0 으로 잘리며 변수 선택이 일어난다')
ax.legend(loc='upper right', frameon=False, fontsize=9, ncol=2)
ax.grid(alpha=0.2, linestyle='--')
plt.show()

In [ ]:
# 최적 λ 에서 살아남은 변수
best = Lasso(alpha=lam_best_l, max_iter=20000).fit(Xs, ys)
print(f'\n라소 (λ = {lam_best_l:.4f}) 의 계수:')
for name, b in zip(features, best.coef_):
    flag = '   ← 제거됨 (계수 = 0)' if abs(b) < 1e-8 else ''
    print(f'  {name:>10s}  {b:+.4f}{flag}')

n_alive = (np.abs(best.coef_) > 1e-8).sum()
print(f'\n살아남은 변수 수: {n_alive} / {len(features)}')

### 4.1 결과 해석

라소가 제거한 변수가 **weight, cups** — 4 부 §5.3 의 표준화 계수 *β\** 가 정확히 0 이었던 두 변수와 일치한다. *rating* 의 공식이 "영양 성분만으로 이루어진 합성 변수" 라는 §5.5 의 발견을, 라소가 자료에서 자동으로 재발견한 셈이다.

## 5. 엘라스틱넷 — L¹ + L²

손실함수.

$$\mathcal{L}_{\text{enet}}(\beta) \;=\; \|y - X\beta\|^2 \;+\; \lambda\,\bigl(\alpha\|\beta\|_1 + (1-\alpha)\|\beta\|_2^2\bigr)$$

두 모수가 있다 — *λ* (전체 강도), *α* (L¹ 비율). *α* = 0 이면 릿지, *α* = 1 이면 라소, 그 사이가 혼합.

In [ ]:
alphas = np.linspace(0.1, 1.0, 10)
lams_en = np.logspace(-3, 0, 18)
grid = np.zeros((len(alphas), len(lams_en)))
for i, a in enumerate(alphas):
    for j, lam in enumerate(lams_en):
        grid[i, j] = -cross_val_score(
            ElasticNet(alpha=lam, l1_ratio=a, max_iter=20000),
            Xs, ys, cv=5, scoring='neg_mean_squared_error').mean()

i_best, j_best = np.unravel_index(np.argmin(grid), grid.shape)
a_best, lam_best_en = alphas[i_best], lams_en[j_best]
print(f'최적 (α, λ) = ({a_best:.2f}, {lam_best_en:.5f})')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.set_facecolor('#FAF6F1')
L_, A_ = np.meshgrid(lams_en, alphas)
pcm = ax.pcolormesh(L_, A_, grid, shading='auto', cmap='viridis_r')
plt.colorbar(pcm, ax=ax, label='5-fold CV MSE')
ax.scatter([lam_best_en], [a_best], marker='*', s=320, c='#C0392B',
           edgecolor='white', linewidth=1.6, zorder=5)
ax.set_xscale('log')
ax.set_xlabel('규제 강도  λ  (로그 축)')
ax.set_ylabel('L¹ 비율  α')
ax.set_title('ElasticNet — (α, λ) 격자에서의 CV 오차')
plt.show()

## 6. 네 모형 한 그림에 — 회귀가족

OLS · Ridge · Lasso · ElasticNet 의 표준화 계수를 함께 본다. 같은 자료가 손실함수의 모양에 따라 어떻게 다르게 풀리는지가 한눈에 들어온다.

In [ ]:
ridge_best = Ridge(alpha=lam_best).fit(Xs, ys).coef_
lasso_best = Lasso(alpha=lam_best_l, max_iter=20000).fit(Xs, ys).coef_
enet_best  = ElasticNet(alpha=lam_best_en, l1_ratio=a_best,
                        max_iter=20000).fit(Xs, ys).coef_

models = {
    'OLS': ols.coef_,
    f'Ridge (λ={lam_best:.2g})': ridge_best,
    f'Lasso (λ={lam_best_l:.2g})': lasso_best,
    f'ENet (α={a_best:.2g}, λ={lam_best_en:.2g})': enet_best,
}

fig, ax = plt.subplots(figsize=(11.5, 5.5))
ax.set_facecolor('#FAF6F1')
xpos = np.arange(len(features))
w = 0.20
cols = ['#C0392B', '#D87B3F', '#D4A437', '#1F7A8C']
for i, ((name, vals), col) in enumerate(zip(models.items(), cols)):
    ax.bar(xpos + (i - 1.5)*w, vals, w, label=name, color=col,
           alpha=0.88, edgecolor='white', linewidth=0.4)
ax.axhline(0, color='#555', linewidth=0.7)
ax.set_xticks(xpos)
ax.set_xticklabels(features, rotation=30, ha='right')
ax.set_ylabel('표준화 계수 β*')
ax.set_title('회귀가족 — 같은 자료, 다른 손실로 풀린 계수들')
ax.legend(loc='lower right', frameon=False, fontsize=9.5, ncol=2)
ax.grid(alpha=0.18, linestyle='--', axis='y')
plt.show()

## 7. 결과 해석 — 다섯 모형의 차이

| 모형 | 다중공선성 처리 | 변수 선택 | 자료에서의 약점 |
|---|---|---|---|
| **OLS** | 처리 안 함 — 계수가 출렁임 | × | *VIF* 큰 변수에서 부호 흔들림 |
| **Ridge** | L² 벌점으로 분산 누름 | × | 0 인 변수도 살려 둠 |
| **Lasso** | L¹ 벌점 + 변수 선택 | ✓ | 묶인 변수 중 하나만 살림 |
| **ElasticNet** | L¹ + L² 혼합 | ✓ | 모수 2 개 — 교차검증 비용 |
| **PCR (6 부)** | 좌표 회전으로 묶임 해소 | △ (PC 단위) | *X* 만 보고 새 변수 만듦 |

*Cereals* 자료에서는 *R²* = 1.0 이라는 자료의 특수성 때문에 — 잡음이 거의 없으므로 — 최적 *λ* 가 매우 작게 잡힌다.  잡음이 큰 자료에서는 *λ* 의 최적값이 더 크게 옮겨가고, 각 모형의 "성격" 이 결과의 차이로 더 분명히 드러난다.

**핵심 한 줄**.  같은 잔차 제곱합 손실에 어떤 벌점을 어떻게 더하는가가 — 회귀 가족의 모든 변주를 결정한다.  자료의 잡음 수준과 변수의 묶임 정도에 따라 — 다섯 가족 중 어느 도구가 적합한지가 갈린다.